# HC18 Out-Of-Fold Post-Hoc Calibration Benchmark

This benchmark evaluates whether post-hoc calibration functions improve Head Circumference (HC) MAE on existing single-fold out-of-fold predictions (`docs/results/hc18/oof_predictions.csv`, $N=999$ images / 806 patients).

### Calibration Protocol
- **Nested 5-Fold Cross-Validation**: Calibration models are fit strictly on 4 training folds and evaluated on held-out fold $k$.
- **Candidate Methods**:
  1. `raw`: Uncorrected model predictions ($y_{pred}$)
  2. `median_add`: Additive shift $y_{pred} + \text{median}(y_{tr} - \hat{y}_{tr})$
  3. `median_mult`: Multiplicative scaling $y_{pred} \times \text{median}(y_{tr} / \hat{y}_{tr})$
  4. `linear_reg`: Linear regression (OLS) $a \cdot y_{pred} + b$
  5. `isotonic`: Non-parametric monotonic regression (`IsotonicRegression(out_of_bounds='clip')`)
- **Statistical Inference**: Patient-level paired bootstrap (5,000 resamples, seed 42) computing 95% CIs of $\Delta \text{MAE}$ vs raw baseline.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# Locate repository root
work_dir = Path.cwd().resolve()
while work_dir.name != 'work' and work_dir.parent != work_dir:
    work_dir = work_dir.parent

results_dir = work_dir / 'docs' / 'results' / 'hc18_calibration'
md_path = results_dir / 'oof_calibration.md'
json_path = results_dir / 'oof_calibration_results.json'
csv_path = results_dir / 'calibrated_predictions.csv'

assert md_path.is_file(), f'Missing {md_path}'
assert json_path.is_file(), f'Missing {json_path}'
assert csv_path.is_file(), f'Missing {csv_path}'

with open(json_path, 'r', encoding='utf-8') as f:
    results = json.load(f)

print('Loaded HC18 OOF Calibration Results JSON.')
print('Probability Map Audit:', results['local_probability_maps_audit']['statement'])
print(f'Total Samples: {results["total_images"]} across {results["total_patients"]} patients')

display(Markdown(md_path.read_text(encoding='utf-8')))

Loaded HC18 OOF Calibration Results JSON.
Probability Map Audit: NO: Zero local OOF probability maps (.npy/.pt) exist in docs/ or data/. Only scalar tabular predictions (oof_predictions.csv) are stored.
Total Samples: 999 across 806 patients


# HC18 Out-Of-Fold Post-Hoc Calibration Study

## Executive Summary
- **Dataset**: HC18 5-fold cross-validation (999 images / 806 patients)
- **Local Probability Maps**: NO: Zero local OOF probability maps (.npy/.pt) exist in docs/ or data/. Only scalar tabular predictions (oof_predictions.csv) are stored.
- **Evaluation Protocol**: Strictly nested 5-fold cross-validation (calibration parameters fit on 4 folds, evaluated on held-out fold).
- **Key Finding**: Raw single-fold models exhibit systematic underestimation (signed bias +1.258 mm under `y_true - y_pred`). Post-hoc calibration reduces HC MAE from **2.302 mm** down to **1.985 mm** (a significant improvement of -0.317 mm, 95% CI [-0.393, -0.240]).

## Overall Calibration Comparison

| Method | MAE (mm) | RMSE (mm) | Median AE (mm) | Signed Bias (True - Pred) Mean / Median | MAE Diff vs Raw [95% CI] (mm) |
|---|---|---|---|---|---|
| Raw (No Correction) | **2.302** | 3.580 | 1.760 | +1.258 / +1.349 | Baseline |
| Median Additive Bias | **2.003** | 3.358 | 1.444 | -0.091 / +0.018 | **-0.299** [-0.374, -0.218] |
| Median Multiplicative Ratio | **1.985** | 3.342 | 1.426 | -0.178 / -0.015 | **-0.317** [-0.404, -0.227] |
| Linear Regression (OLS) | **1.985** | 3.335 | 1.427 | -0.002 / +0.154 | **-0.317** [-0.393, -0.240] |
| Isotonic Regression | **2.184** | 3.566 | 1.523 | -0.028 / +0.031 | **-0.118** [-0.227, -0.005] |

## Per-Tercile MAE Analysis (Head-Size Proxy for Gestational Age)

- **Low Tercile**: 45.1 – 167.2 mm ($N=333$)
- **Mid Tercile**: 167.5 – 183.0 mm ($N=333$)
- **High Tercile**: 183.0 – 349.3 mm ($N=333$)

| Method | Low Tercile MAE (mm) | Mid Tercile MAE (mm) | High Tercile MAE (mm) | Overall MAE (mm) |
|---|---|---|---|---|
| Raw (No Correction) | 1.917 | 2.023 | 2.966 | 2.302 |
| Median Additive Bias | 1.798 | 1.618 | 2.592 | 2.003 |
| Median Multiplicative Ratio | 1.758 | 1.628 | 2.568 | 1.985 |
| Linear Regression (OLS) | 1.753 | 1.638 | 2.562 | 1.985 |
| Isotonic Regression | 2.080 | 1.673 | 2.798 | 2.184 |

## Explicit Caveats & Integrity Guards

1. **Single-Fold vs Ensemble**: This study evaluates single-fold out-of-fold predictions. The official HC18 challenge test submission used a 5-fold ensemble average of predicted ellipse parameters. Averaging across 5 independent fold models already reduces prediction variance and naturally shifts predictions toward the ensemble centroid.
2. **Signed Bias Conventions**:
   - **Convention A (`y_true - y_pred`)**: Raw mean = +1.2576 mm, median = +1.3490 mm (positive indicates underprediction).
   - **Convention B (`y_pred - y_true`)**: Raw mean = -1.2576 mm, median = -1.3490 mm (negative indicates underprediction).
3. **Submission Immutability**: Zero changes are made to the official challenge submission CSVs (submission_ensemble_5fold.csv and submission_single_fold1.csv) or submission generation scripts. This audit is strictly observational on the existing held-out cross-validation records.

In [2]:
# Summary Table of Statistical Differences vs Baseline
rows = []
for m, met in results['methods_metrics'].items():
    b_res = results['bootstrap_diff_vs_raw'][m]
    rows.append({
        'Method': m,
        'MAE (mm)': round(met['mae'], 4),
        'RMSE (mm)': round(met['rmse'], 4),
        'Median AE (mm)': round(met['median_ae'], 4),
        'Signed Bias Mean (mm)': round(met['signed_bias_true_minus_pred_mean'], 4),
        'Mean Diff vs Raw (mm)': round(b_res['mean_diff_vs_raw'], 4),
        '95% CI Lower (mm)': round(b_res['ci_95'][0], 4),
        '95% CI Upper (mm)': round(b_res['ci_95'][1], 4),
        'Significant (CI < 0)': b_res['significant_improvement'],
    })
df_summary = pd.DataFrame(rows)
display(df_summary)

,Method,MAE (mm),RMSE (mm),Median AE (mm),Signed Bias Mean (mm),Mean Diff vs Raw (mm),95% CI Lower (mm),95% CI Upper (mm),Significant (CI < 0)
0,raw,2.3019,3.5797,1.7603,1.2576,0.0000,0.0000,0.0000,False
1,median_add,2.0029,3.3579,1.4441,-0.0909,-0.2990,-0.3738,-0.2185,True
2,median_mult,1.9846,3.3418,1.4258,-0.1779,-0.3173,-0.4036,-0.2270,True
3,linear_reg,1.9846,3.3350,1.4273,-0.0023,-0.3174,-0.3928,-0.2398,True
4,isotonic,2.1836,3.5656,1.5233,-0.0279,-0.1184,-0.2272,-0.0048,True


In [3]:
# Calibrated Predictions Sample
df_cal = pd.read_csv(csv_path)
print(f'Calibrated predictions table: {len(df_cal)} rows')
display(df_cal[['image', 'fold', 'tercile', 'true_HC_mm', 'pred_HC_raw', 'pred_HC_linear_reg', 'err_HC_raw', 'err_HC_linear_reg']].head(10))

Calibrated predictions table: 999 rows


,image,fold,tercile,true_HC_mm,pred_HC_raw,pred_HC_linear_reg,err_HC_raw,err_HC_linear_reg
0,002_HC.png,1,low,69.3877,67.4633,68.028690,1.9244,1.359010
1,007_HC.png,1,low,63.9141,62.1593,62.691104,1.7548,1.222996
2,010_2HC.png,1,low,73.6951,70.9286,71.515933,2.7665,2.179167
3,010_HC.png,1,low,72.7561,72.6404,73.238573,0.1157,0.482473
4,023_2HC.png,1,low,69.2311,69.3726,69.950080,0.1415,0.718980
5,023_HC.png,1,low,71.1518,68.3312,68.902086,2.8206,2.249714
6,029_HC.png,1,low,66.9567,66.3706,66.929071,0.5861,0.027629
7,030_HC.png,1,low,64.8590,64.0811,64.625073,0.7779,0.233927
8,031_HC.png,1,low,67.8553,65.2229,65.774103,2.6324,2.081197
9,033_2HC.png,1,low,79.8648,78.2270,78.860549,1.6378,1.004251


In [4]:
# Verify Explicit Caveats
print('=== EXPLICIT CAVEATS & INTEGRITY CHECKS ===')
for k, v in results['caveats'].items():
    print(f'\n[{k.upper()}]\n{v}')

=== EXPLICIT CAVEATS & INTEGRITY CHECKS ===

[CAVEAT_1_SINGLE_FOLD_VS_ENSEMBLE]
This study evaluates single-fold out-of-fold predictions. The official HC18 challenge test submission used a 5-fold ensemble average of predicted ellipse parameters. Averaging across 5 independent fold models already reduces prediction variance and naturally shifts predictions toward the ensemble centroid.

[CAVEAT_2_SIGNED_DIFFERENCE_CONVENTION]
Both signed difference conventions are explicitly reported: (1) Under (y_true - y_pred), raw predictions have a mean signed bias of +1.2576 mm (median +1.3490 mm), confirming that the single-fold models systematically underestimated ground-truth HC. (2) Under (y_pred - y_true), raw signed mean is -1.2576 mm (median -1.3490 mm).

[CAVEAT_3_NO_CHALLENGE_SUBMISSION_CHANGES]
Zero changes are made to the official challenge submission CSVs (submission_ensemble_5fold.csv and submission_single_fold1.csv) or submission generation scripts. This audit is strictly observationa